<a href="https://colab.research.google.com/github/MohammadJoenathan/TextMining_PenerapanTextVectorizer/blob/main/2218060MohammadJoenathanApp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Import semua modul Library yang dibutuhkan

In [1]:
!pip install Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 4.7 MB/s eta 0:00:00


In [2]:
!pip install Sastrawi scikit-learn joblib

In [3]:
import json
import joblib
import numpy as np
import re
from sklearn.metrics.pairwise import euclidean_distances
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# 2. Menghubungkan Google Colab dengan Google Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 3. Query / Testing (Pencarian Jawaban) Menggunakan Vector Database

In [5]:
# Load stemmer bahasa indonesia
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Fungsi preprocessing text untuk membersihkan teks sebelum dihitung tf-idf
def preprocess_text(text):
    text = text.lower()  # Mengubah semua huruf menjadi kecil
    text = re.sub(r'[^\w\s]', '', text)  # Menghapus simbol dan tanda baca
    words = text.split()  # Memecah kalimat menjadi kata-kata
    stemmed_words = [stemmer.stem(word) for word in words]  # Stemming setiap kata
    return " ".join(stemmed_words)  # Menggabungkan kembali menjadi kalimat

# Load model tf-idf vectorizer dari google drive
vectorizer = joblib.load("/content/drive/MyDrive/Colab Notebooks/TextMiningUTS/vectorizer.joblib")

# Load vector database dari vector.json
with open("/content/drive/MyDrive/Colab Notebooks/TextMiningUTS/vector.json", "r", encoding="utf-8") as file:
    question_vectors = json.load(file)

# Threshold
# Jika jarak lebih besar dari threshold, maka dianggap tidak cocok
THR = 1.0

# Fungsi untuk mencari pertanyaan paling mirip berdasarkan euclidean distance
def find_best_match(user_question):
    user_question_processed = preprocess_text(user_question)  # Preprocessing input user
    user_vector = vectorizer.transform([user_question_processed]).toarray()  # Ubah ke vektor tf-idf

    closest_distance = float("inf")  # Nilai awal jarak terdekat
    closest_question = None
    closest_answer = None

    # Loop semua data dalam vector database
    for item in question_vectors:
        question_vector = np.array(item["vector"]).reshape(1, -1)  # Ambil vector dari database
        distance = euclidean_distances(user_vector, question_vector)[0][0]

        # Simpan pertanyaan dengan jarak paling kecil dan masih di bawah threshold
        if distance < closest_distance and distance <= THR:
            closest_distance = distance
            closest_question = item["question"]
            closest_answer = item.get("answer", "Maaf, tidak ada jawaban yang tersedia.")

    return closest_question, closest_answer, closest_distance

# Program utama sistem tanya jawab
print("Selamat datang di sistem tanya jawab timnas sepak bola indonesia.")
print("Ketik 'exit' untuk keluar.\n")

while True:
    user_question = input("Silahkan bertanya: ")

    # Keluar jika user mengetik exit
    if user_question.lower() == "exit":
        print("Terima kasih telah menggunakan sistem tanya jawab timnas indonesia.")
        break

    closest_question, closest_answer, closest_distance = find_best_match(user_question)

    # Jika ditemukan pertanyaan yang sesuai
    if closest_question:
        print(f"\nPertanyaan terkait: {closest_question}")
        print(f"Timnas bot: {closest_answer}")
        print(f"Distance score: {closest_distance}\n")
    else:
        print("\nMaaf, tidak menemukan jawaban yang sesuai dengan pertanyaan anda.\n")

Selamat datang di sistem tanya jawab timnas sepak bola indonesia.
Ketik 'exit' untuk keluar.

Silahkan bertanya: Apa julukan Timnas Indonesia

Pertanyaan terkait: Apa julukan Timnas Indonesia?
Timnas bot: Julukan Timnas Indonesia adalah Garuda.
Distance score: 0.0

Silahkan bertanya: Siapa organisasi yang menaungi Timnas Indonesia

Pertanyaan terkait: Siapa organisasi yang menaungi Timnas Indonesia?
Timnas bot: Timnas Indonesia dinaungi oleh PSSI (Persatuan Sepak Bola Seluruh Indonesia).
Distance score: 0.0

Silahkan bertanya: Apa itu FIFA Matchday

Pertanyaan terkait: Apa itu FIFA Matchday?
Timnas bot: FIFA Matchday adalah jadwal resmi FIFA untuk pertandingan internasional di mana klub wajib melepas pemainnya.
Distance score: 0.0

Silahkan bertanya: exit
Terima kasih telah menggunakan sistem tanya jawab timnas indonesia.
